# Calibración del Autómata Celular con Caso de Estudio Histórico

Este notebook integra las dos fuentes de datos de la Etapa 1:
1. **NASA FIRMS:** Focos de calor satelitales en la selva peruana (Madre de Dios / Ucayali).
2. **Open-Meteo ERA5 Reanalysis:** Serie horaria de velocidad y dirección del viento histórico para las fechas del evento.
3. **Autómata Celular con Viento:** Simulación paso a paso con el módulo `WindField` de Alexandridis et al.

In [ ]:
import os
import sys
from pathlib import Path

# Asegurar que la raíz del proyecto esté en el path
PROJECT_ROOT = Path("..").resolve() if Path("src").exists() is False else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.firms_client import FIRMSClient
from src.data.weather_client import WeatherClient
from src.model.grid import Grid, SANA, QUEMANDOSE, QUEMADA
from src.model.wind_influence import WindField, get_moore_wind_factors
print("Módulos importados correctamente.")

## 1. Extracción de Viento Histórico Horario (ERA5)

Consultamos la serie horaria para las coordenadas del incendio (Puerto Maldonado: `-12.5933, -69.1891`) durante el pico de incendios de agosto de 2024.

In [ ]:
weather_client = WeatherClient()
lat, lon = -12.5933, -69.1891
start_date, end_date = "2024-08-14", "2024-08-16"

serie_viento = weather_client.get_hourly_wind_series(
    lat=lat,
    lon=lon,
    start_date=start_date,
    end_date=end_date,
)

print(f"Horas de datos obtenidas: {serie_viento['total_hours']}")
print(f"Fuente: {serie_viento['source']}")
print("Primeras 3 horas registradas:")
for h in serie_viento["series"][:3]:
    print(f"  {h['time_utc']} -> Viento: {h['speed_ms']} m/s, Dirección: {h['deg']}° ({h['cardinal']}), Empuje Fuego: {h['propagation_vector']['travel_deg']}°")

## 2. Acoplamiento del Viento con el Autómata Celular

Instanciamos una grilla sintética de prueba de 50x50 y ejecutamos la simulación hora por hora, actualizando en cada paso el `WindField` con los datos meteorológicos reales.

In [ ]:
grid = Grid(tamano=50, prob_ignicion_base=0.4, pasos_para_quemarse=3, semilla=42)
centro = grid.tamano // 2
grid.encender_celda(centro, centro)

viento_field = WindField()

print("Evolución del incendio a lo largo de las primeras 12 horas:")
for i, hora in enumerate(serie_viento["series"][:12]):
    # Actualizar condiciones del viento en este paso de tiempo
    viento_field.update_wind(speed_ms=hora["speed_ms"], wind_deg=hora["deg"])
    
    # Avanzar el autómata celular modulado por el viento
    grid.paso_tiempo(campo_viento=viento_field)
    
    estados = grid.contar_estados()
    print(f"Hora {i+1:02d} ({hora['time_utc']}) | Viento: {hora['speed_ms']} m/s ({hora['cardinal']}) | Quemandose: {estados['Quemandose']:3d} | Quemadas: {estados['Quemada']:3d}")

## 3. Conclusiones de la Integración
- El modelo es capaz de ingerir dinámicamente series horarias de reanálisis ERA5 sin interrupción de cuotas.
- La formulación matemática de Alexandridis acelera el avance hacia la dirección de empuje del viento y frena el avance en contra.
- Próximo paso: Reemplazar el grid sintético por los recortes reales de vegetación (ESA WorldCover) y elevación (SRTM) en el Sprint 2.